In [8]:
import pandas as pd  # 데이터 처리
import numpy as np  # 수치 처리
import re  # 정규표현식


# ===== 경로 =====
PATH_SUFUL = r"C:/Users/User/OneDrive/문서/3yejoo/ERP/재고/251231_재고조사/251231_본사재고수불부모음_ver6_수정.xlsx"  # 수불부 경로
PATH_SUBMIT = r"C:/Users/User/OneDrive/문서/3yejoo/ERP/재고/251231_재고조사/251231_본사제출용.xlsx"  # 제출본 경로
OUT_XLSX = r"./대체이력_재고수불부_보정본_최종.xlsx"  # 결과 저장 경로

# ===== 옵션 =====
WAREHOUSE_KEYWORD = "본사"  # 본사만 필터
TOL_1 = 0.20  # 1차 단가 허용폭
TOL_2 = 0.30  # 2차 단가 허용폭

# ===== 컬럼명(수불부) =====
DATE_COL = "일자"  # 날짜
ITEM_COL = "품목코드"  # 품목코드
NAME_COL = "품목명"  # 품목명
STOCK_COL = "재고수량"  # 재고수량
OUT_QTY_COL = "출고수량"  # 출고수량
OUT_PRICE_COL = "출고단가"  # 출고단가
WH_COL = "창고명"  # 창고명
PARTY_COL = "거래처명"  # 거래처명

# ===== 컬럼명(제출본) =====
SUBMIT_STOCK_COL = "본사실재고"  # 제출본 실재고
SUBMIT_BEFORE_COL = "본사\n실-전"  # 제출본 본사실-전(대체가능 조건)
SUBMIT_USE_COL = "사용여부"  # 제출본 사용여부
SUBMIT_NAME_COL = "품목명"  # 제출본 품목명

# ===== 함수: 품목코드 정리 =====
def normalize_item_code(code: str) -> str:  # 품목코드 표준화
    s = str(code).strip()  # 공백 제거
    s = re.sub(r"\.0$", "", s)  # 끝 .0 제거
    if re.fullmatch(r"\d+", s):  # 숫자만이면
        s = s.lstrip("0")  # 앞 0 제거
        return s if s != "" else "0"  # 전부 0이면 0
    return s  # 문자 포함은 그대로

# ===== 로드 =====
df_submit = pd.read_excel(PATH_SUBMIT)  # 제출본 로드
df_suful = pd.read_excel(PATH_SUFUL)  # 수불부 로드

In [ ]:

SUBMIT_BEFORE_COL = "본사\n실-전"  # 제출본 본사실-전(대체가능 조건)
# ===== 전처리: 수불부 =====
df_suful[DATE_COL] = df_suful[DATE_COL].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()  # 날짜 공백 정리
df_suful[DATE_COL] = pd.to_datetime(df_suful[DATE_COL], errors="coerce", format="mixed")  # 날짜 변환
df_suful = df_suful.dropna(subset=[DATE_COL]).copy()  # 날짜 없는 행 제거
df_suful[ITEM_COL] = df_suful[ITEM_COL].astype(str).apply(normalize_item_code)  # 품목코드 정리
df_suful[STOCK_COL] = pd.to_numeric(df_suful[STOCK_COL], errors="coerce").fillna(0).astype(float)  # 재고수량 숫자화
df_suful[OUT_QTY_COL] = pd.to_numeric(df_suful[OUT_QTY_COL], errors="coerce").fillna(0).astype(float)  # 출고수량 숫자화
df_suful[OUT_PRICE_COL] = pd.to_numeric(df_suful[OUT_PRICE_COL], errors="coerce").fillna(0).astype(float)  # 출고단가 숫자화
df_suful = df_suful[df_suful[WH_COL].astype(str).str.contains(WAREHOUSE_KEYWORD, na=False)].copy()  # 본사만
df_suful = df_suful[df_suful[PARTY_COL].astype(str).str.strip() != "[조정]"].copy()  # [조정] 제외
df_suful = df_suful.sort_values([ITEM_COL, DATE_COL]).reset_index(drop=True)  # 정렬

# ===== 전처리: 제출본 =====
df_submit[ITEM_COL] = df_submit[ITEM_COL].astype(str).apply(normalize_item_code)  # 품목코드 정리
df_submit[SUBMIT_STOCK_COL] = pd.to_numeric(df_submit[SUBMIT_STOCK_COL], errors="coerce").fillna(0).astype(float)  # 실재고 숫자화
df_submit[SUBMIT_BEFORE_COL] = pd.to_numeric(df_submit[SUBMIT_BEFORE_COL], errors="coerce").fillna(0).astype(float)  # 본사실-전 숫자화
df_submit[SUBMIT_USE_COL] = df_submit[SUBMIT_USE_COL].astype(str).str.strip()  # 사용여부 공백 제거
df_submit = df_submit.copy()  # 안전 복사

# ===== 제출본 빠른 조회 맵 =====
submit_map = df_submit.set_index(ITEM_COL, drop=False)  # 품목코드 인덱스화

# ===== 공통 함수: 제출본 1행 =====
def get_submit_row(code: str) -> pd.Series:  # 제출본 1행 조회
    if code not in submit_map.index:  # 없으면
        return pd.Series(dtype=object)  # 빈값
    return submit_map.loc[code]  # 1행 반환

# ===== 공통 함수: 기준일 재고(직전 포함) =====
def stock_asof(df: pd.DataFrame, code: str, date: pd.Timestamp) -> float:  # 기준일 재고
    d = df[(df[ITEM_COL] == code) & (df[DATE_COL] <= date)].copy()  # 기준일 이전/당일
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    return float(pd.to_numeric(d[STOCK_COL], errors="coerce").fillna(np.nan).iloc[-1])  # 마지막 재고

# ===== 공통 함수: 기준일 이후 최소재고 =====
def min_stock_after(df: pd.DataFrame, code: str, date: pd.Timestamp) -> float:  # 기준일 이후 최소재고
    d = df[(df[ITEM_COL] == code) & (df[DATE_COL] >= date)].copy()  # 이후만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    return float(pd.to_numeric(d[STOCK_COL], errors="coerce").min())  # 최소값

# ===== 공통 함수: 당일 출고단가(유효 출고만 평균) =====
def out_price_on_date(df: pd.DataFrame, code: str, date: pd.Timestamp) -> float:  # 당일 단가
    d = df[(df[ITEM_COL] == code) & (df[DATE_COL] == date)].copy()  # 당일만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d[OUT_QTY_COL] = pd.to_numeric(d[OUT_QTY_COL], errors="coerce").fillna(0)  # 출고수량
    d[OUT_PRICE_COL] = pd.to_numeric(d[OUT_PRICE_COL], errors="coerce")  # 출고단가
    d = d[(d[OUT_QTY_COL] > 0) & (d[OUT_PRICE_COL] > 0)].copy()  # 유효만
    if len(d) == 0:  # 유효 없으면
        return np.nan  # 결측
    return float(d[OUT_PRICE_COL].mean())  # 평균 단가

def out_price_in_window(df: pd.DataFrame, code: str, center: pd.Timestamp, days: int = 7) -> float:  # +-기간 단가
    start = center - pd.Timedelta(days=days)  # 시작일
    end = center + pd.Timedelta(days=days)  # 종료일
    d = df[(df[ITEM_COL] == code) & (df[DATE_COL] >= start) & (df[DATE_COL] <= end)].copy()  # 기간 필터
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d[OUT_QTY_COL] = pd.to_numeric(d[OUT_QTY_COL], errors="coerce").fillna(0)  # 출고수량
    d[OUT_PRICE_COL] = pd.to_numeric(d[OUT_PRICE_COL], errors="coerce")  # 출고단가
    d = d[(d[OUT_QTY_COL] > 0) & (d[OUT_PRICE_COL] > 0)].copy()  # 유효만
    if len(d) == 0:  # 유효 없으면
        return np.nan  # 결측
    d["dist"] = (d[DATE_COL] - center).abs()  # 기준일과 거리
    d = d.sort_values(["dist", DATE_COL]).copy()  # 가까운 순
    return float(d[OUT_PRICE_COL].iloc[0])  # 가장 가까운 단가

# ===== 공통 함수: 기준일 이후 가장 가까운 유효 출고단가 =====
def nearest_valid_out_price_after(df: pd.DataFrame, code: str, date: pd.Timestamp) -> float:  # 기준단가
    d = df[(df[ITEM_COL] == code) & (df[DATE_COL] >= date)].copy()  # 이후만
    if len(d) == 0:  # 없으면
        return np.nan  # 결측
    d[OUT_QTY_COL] = pd.to_numeric(d[OUT_QTY_COL], errors="coerce").fillna(0)  # 출고수량
    d[OUT_PRICE_COL] = pd.to_numeric(d[OUT_PRICE_COL], errors="coerce")  # 출고단가
    d = d[(d[OUT_QTY_COL] > 0) & (d[OUT_PRICE_COL] > 0)].copy()  # 유효만
    if len(d) == 0:  # 유효 없으면
        return np.nan  # 결측
    d = d.sort_values(DATE_COL).copy()  # 날짜 정렬
    return float(d[OUT_PRICE_COL].iloc[0])  # 가장 가까운 단가

# ===== 공통 함수: 거래처명(이력용) =====
def pick_party(df: pd.DataFrame, code: str, date: pd.Timestamp) -> str:  # 거래처명
    d = df[(df[ITEM_COL] == code) & (df[DATE_COL] == date)].copy()  # 당일 행
    if len(d) == 0:  # 없으면
        return "[대체보정]"  # 기본값
    v = str(d.iloc[0].get(PARTY_COL, "")).strip()  # 거래처명
    return v if v != "" and v.lower() != "nan" else "[대체보정]"  # 빈값 처리

# ===== 공통 함수: 첫 마이너스 구간(보정기준일/보정수량) =====
def first_minus_fix(df: pd.DataFrame, code: str):  # 첫 마이너스 구간 계산
    d = df[df[ITEM_COL] == code].copy()  # 해당 품목만
    if len(d) == 0:  # 없으면
        return None  # 없음
    d = d.sort_values(DATE_COL).reset_index(drop=True)  # 날짜 정렬
    d["is_minus"] = pd.to_numeric(d[STOCK_COL], errors="coerce").fillna(0) < 0  # 마이너스 여부
    if d["is_minus"].sum() == 0:  # 마이너스 없으면
        return None  # 없음
    d["group"] = d["is_minus"].ne(d["is_minus"].shift()).cumsum()  # 상태변화 구간
    dm = d[d["is_minus"]].copy()  # 마이너스만
    g = dm.groupby("group").agg(시작일=(DATE_COL, "min"), 최대마이너스=(STOCK_COL, "min")).reset_index(drop=True)  # 구간 요약
    first = g.sort_values("시작일").iloc[0]  # 첫 마이너스 구간
    target_date = pd.Timestamp(first["시작일"])  # 보정기준일
    need_qty = int(abs(float(first["최대마이너스"])))  # 보정수량
    return target_date, need_qty  # 반환

# ===== 공통 함수: 후보 생성 + 분할 이동계획 생성 =====
def build_moves(df_adj: pd.DataFrame, target_code: str, target_date: pd.Timestamp, need_qty: int, tol: float):  # 분할 이동계획
    sr = get_submit_row(target_code)  # 제출본 1행
    if len(sr) == 0:  # 제출본 없으면
        return []  # 실패
    target_use = str(sr.get(SUBMIT_USE_COL, "")).strip()  # 사용여부
    target_name = str(sr.get(SUBMIT_NAME_COL, "")).strip()  # 품목명
    base_price = nearest_valid_out_price_after(df_adj, target_code, target_date)  # 문제품목 기준단가
    if pd.isna(base_price) or float(base_price) <= 0:  # 단가 없으면
        return []  # 실패

    cand = df_submit.copy()  # 후보 시작(제출본 전체)
    cand = cand[cand[ITEM_COL].astype(str) != str(target_code)].copy()  # 자기 제외
    cand = cand[pd.to_numeric(cand[SUBMIT_BEFORE_COL], errors="coerce").fillna(0) > 0].copy()  # (필수1) 본사실-전 양수
    cand = cand[cand[SUBMIT_USE_COL].astype(str).str.strip() == target_use].copy()  # (필수2) 사용여부 동일

    cand["T_출고단가"] = cand[ITEM_COL].apply(lambda x: out_price_in_window(df_adj, str(x), target_date, 7))  # +-7일 단가
    cand = cand[~pd.isna(cand["T_출고단가"])].copy()  # 단가 없으면 제외

    cand["단가차이율"] = (cand["T_출고단가"] - float(base_price)).abs() / float(base_price)  # 단가차이율
    cand = cand[cand["단가차이율"] <= float(tol)].copy()  # 허용폭 필터

    cand["T이후_최소재고"] = cand[ITEM_COL].apply(lambda x: min_stock_after(df_adj, str(x), target_date))  # 이후 최소재고
    cand = cand[~pd.isna(cand["T이후_최소재고"])].copy()  # 최소재고 없으면 제외

    cand["가용이동량"] = cand["T이후_최소재고"].apply(lambda x: int(np.floor(max(0, float(x)))))  # 마이너스 없이 뺄 수 있는 최대량
    cand = cand[cand["가용이동량"] > 0].copy()  # 1개라도 가능해야 함

    cand = cand.sort_values(["단가차이율", "가용이동량"], ascending=[True, False]).reset_index(drop=True)  # 좋은 후보 우선

    need = int(need_qty)  # 남은 필요수량
    moves = []  # 이동계획

    for i in range(len(cand)):  # 후보 순회
        if need <= 0:  # 다 채웠으면
            break  # 종료
        sub_code = str(cand.loc[i, ITEM_COL])  # 대체품목코드
        sub_name = str(cand.loc[i, SUBMIT_NAME_COL])  # 대체품목명
        avail = int(cand.loc[i, "가용이동량"])  # 가용이동량
        mv = int(min(need, avail))  # 배정 이동량
        if mv <= 0:  # 0이면
            continue  # 스킵
        moves.append({"대체품목코드": sub_code, "대체품목명": sub_name, "이동량": mv})  # 이동 추가
        need -= mv  # 남은 수량 감소

    if need > 0:  # 다 못 채우면
        return []  # 실패

    return moves  # 성공

# ===== 공통 함수: 보정 적용(재고수량만 보정) =====
def apply_adjustment(df_adj: pd.DataFrame, target_code: str, target_date: pd.Timestamp, moves: list, need_qty: int) -> pd.DataFrame:  # 보정 반영
    out = df_adj.copy()  # 복사
    mt = (out[ITEM_COL] == target_code) & (out[DATE_COL] >= target_date)  # 문제품목 이후
    out.loc[mt, STOCK_COL] = pd.to_numeric(out.loc[mt, STOCK_COL], errors="coerce").fillna(0) + float(need_qty)  # 문제품목 +보정
    for m in moves:  # 대체품목별
        sub_code = str(m["대체품목코드"])  # 코드
        qty = float(m["이동량"])  # 수량
        ms = (out[ITEM_COL] == sub_code) & (out[DATE_COL] >= target_date)  # 대체품목 이후
        out.loc[ms, STOCK_COL] = pd.to_numeric(out.loc[ms, STOCK_COL], errors="coerce").fillna(0) - qty  # 대체품목 -보정
    out = out.sort_values([ITEM_COL, DATE_COL]).reset_index(drop=True)  # 정렬
    return out  # 반환

# ===== 공통 함수: 해결 여부(기준일 이후 마이너스 없음) =====
def is_solved(df_adj: pd.DataFrame, code: str, date: pd.Timestamp) -> bool:  # 해결 체크
    smin = min_stock_after(df_adj, code, date)  # 이후 최소재고
    if pd.isna(smin):  # 없으면
        return True  # 문제 없음 취급
    return float(smin) >= 0  # 0 이상이면 해결

# ===== 공통 함수: 대체품목 마이너스 방지(기준일 이후 최소재고 0 이상) =====
def subs_ok(df_adj: pd.DataFrame, moves: list, date: pd.Timestamp) -> bool:  # 대체품목 검증
    for m in moves:  # 이동 목록
        sub_code = str(m["대체품목코드"])  # 대체품목코드
        smin = min_stock_after(df_adj, sub_code, date)  # 이후 최소재고
        if pd.isna(smin) or float(smin) < 0:  # 마이너스면
            return False  # 실패
    return True  # 성공

# ===== 처리 대상: 수불부에 존재하는 품목들 =====
all_items = df_suful[ITEM_COL].dropna().astype(str).unique().tolist()  # 전체 품목코드

# ===== 누적 결과 =====
df_suful_adj = df_suful.copy()  # 보정 누적용
move_logs = []  # 이력 누적
summary_rows = []  # 요약 누적

# ===== 메인 루프 =====
for target_code in all_items:  # 모든 품목 순회
    fix = first_minus_fix(df_suful_adj, target_code)  # 첫 마이너스 구간 찾기
    if fix is None:  # 마이너스 없으면
        sr0 = get_submit_row(target_code)  # 제출본 1행
        nm0 = str(sr0.get(SUBMIT_NAME_COL, "")).strip() if len(sr0) > 0 else ""  # 품목명
        summary_rows.append({"문제품목코드": target_code, "문제품목명": nm0, "보정기준일": "", "보정수량": 0, "대체내역": "", "결과": "해당없음", "미해결": ""})  # 요약
        continue  # 다음 품목

    target_date, need_qty = fix  # 보정기준일/보정수량
    sr = get_submit_row(target_code)  # 제출본 1행
    target_name = str(sr.get(SUBMIT_NAME_COL, "")).strip() if len(sr) > 0 else ""  # 품목명
    party = pick_party(df_suful_adj, target_code, target_date)  # 거래처명(이력용)

    final_moves = []  # 최종 이동계획
    final_tol = None  # 최종 허용폭
    result_tag = ""  # 결과 태그

    moves_18 = build_moves(df_suful_adj, target_code, target_date, need_qty, TOL_1)  # 18% 시도
    if len(moves_18) > 0:  # 계획 있으면
        df_try = apply_adjustment(df_suful_adj, target_code, target_date, moves_18, need_qty)  # 보정 적용
        if is_solved(df_try, target_code, target_date) and subs_ok(df_try, moves_18, target_date):  # 둘 다 통과면
            final_moves = moves_18  # 확정
            final_tol = TOL_1  # 허용폭
            result_tag = "18%"  # 결과
            df_suful_adj = df_try  # 누적 반영

    if result_tag == "":  # 18% 실패면
        moves_25 = build_moves(df_suful_adj, target_code, target_date, need_qty, TOL_2)  # 25% 시도
        if len(moves_25) > 0:  # 계획 있으면
            df_try = apply_adjustment(df_suful_adj, target_code, target_date, moves_25, need_qty)  # 보정 적용
            if is_solved(df_try, target_code, target_date) and subs_ok(df_try, moves_25, target_date):  # 둘 다 통과면
                final_moves = moves_25  # 확정
                final_tol = TOL_2  # 허용폭
                result_tag = "25%"  # 결과
                df_suful_adj = df_try  # 누적 반영

    if result_tag == "":  # 둘 다 실패면
        summary_rows.append({"문제품목코드": target_code, "문제품목명": target_name, "보정기준일": str(target_date.date()), "보정수량": int(need_qty), "대체내역": "", "결과": "미해결", "미해결": "미해결"})  # 요약
        print(f"{target_code} | {target_date.date()} | {need_qty} |  | 미해결")  # 1줄 출력
        continue  # 다음 품목

    target_stock_before = stock_asof(df_suful_adj, target_code, target_date) - float(need_qty)  # 문제품목 재고(보정 전 추정)
    replace_text_parts = []  # 출력용 대체내역
    for m in final_moves:  # 이동별 이력
        sub_code = str(m["대체품목코드"])  # 대체품목코드
        sub_name = str(m["대체품목명"])  # 대체품목명
        qty = float(m["이동량"])  # 이동량
        sub_before = stock_asof(df_suful_adj, sub_code, target_date) + qty  # 대체품목 재고(보정 전 추정)
        move_logs.append({  # 이력 1줄
            "일자": target_date,  # 일자
            "거래처": party,  # 거래처
            "문제품목코드": target_code,  # 문제품목코드
            "문제품목명": target_name,  # 문제품목명
            "문제품목재고량": float(target_stock_before),  # 문제품목재고량(보정 전)
            "대체품목코드": sub_code,  # 대체품목코드
            "대체품목명": sub_name,  # 대체품목명
            "대체품목재고량": float(sub_before),  # 대체품목재고량(보정 전)
            "이동량": float(qty),  # 이동량
            "대체품목이동후 재고량": float(sub_before) - float(qty),  # 이동 후 재고
        })  # 로그 추가
        replace_text_parts.append(f"{sub_code}({int(qty)})")  # 출력용 문자열

    replace_text = ",".join(replace_text_parts)  # 출력용 대체내역
    summary_rows.append({"문제품목코드": target_code, "문제품목명": target_name, "보정기준일": str(target_date.date()), "보정수량": int(need_qty), "대체내역": replace_text, "결과": f"성공({result_tag})", "미해결": ""})  # 요약
    print(f"{target_code} | {target_date.date()} | {need_qty} | {replace_text} | 성공({result_tag})")  # 1줄 출력

# ===== 결과 DF 생성 =====
df_move_log = pd.DataFrame(move_logs)  # 이력 DF
df_summary = pd.DataFrame(summary_rows)  # 요약 DF

# ===== 저장 =====
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:  # 엑셀 저장
    df_move_log.to_excel(writer, sheet_name="이력이력", index=False)  # 이력 시트
    df_suful_adj.to_excel(writer, sheet_name="재고수불부_보정본", index=False)  # 보정본 시트
    df_summary.to_excel(writer, sheet_name="요약", index=False)  # 요약 시트


1 | 2025-09-04 | 1 | 1337(1) | 성공(18%)
10 | 2025-05-29 | 3 |  | 미해결
1002 | 2025-02-13 | 1 |  | 미해결
1003 | 2025-01-02 | 10 |  | 미해결
1010 | 2025-05-07 | 2 |  | 미해결
1017 | 2025-01-03 | 2 |  | 미해결
1018 | 2025-08-20 | 8 |  | 미해결
1019 | 2025-01-14 | 3 |  | 미해결
1029 | 2025-04-23 | 1 | 1132(1) | 성공(18%)
1030 | 2025-12-11 | 1 | 3420(1) | 성공(18%)
1031 | 2025-05-26 | 2 |  | 미해결
1038 | 2025-02-21 | 8 | 870(8) | 성공(18%)
1043 | 2025-05-07 | 34 | 1343(12),2372(3),2401(4),14(4),1337(1),562(10) | 성공(18%)
1046 | 2025-03-07 | 20 |  | 미해결
1048 | 2025-01-03 | 34 | 1415(18),1750(15),1604(1) | 성공(18%)
1053 | 2025-03-26 | 2 |  | 미해결
1054 | 2025-01-02 | 2 |  | 미해결
1055 | 2025-01-08 | 1 |  | 미해결
1071 | 2025-02-17 | 1 | 1412(1) | 성공(18%)
1076 | 2025-04-08 | 7 |  | 미해결
1080 | 2025-08-26 | 27 | 14(27) | 성공(18%)
1082 | 2025-01-08 | 8 | 951(8) | 성공(25%)
1088 | 2025-04-24 | 61 | 870(14),3069(2),4688(45) | 성공(18%)
1090 | 2025-06-04 | 1 |  | 미해결
1100 | 2025-05-08 | 17 | 263(17) | 성공(18%)
1101 | 2025-05-08 | 7 | 263(7) 

: 

In [6]:
for i, col in enumerate(df_submit.columns):  # 컬럼 순회
    print(i, repr(col))  # 인덱스 + 실제 컬럼명(공백 포함) 출력

0 '품목코드'
1 '품목명'
2 '단위'
3 '규격'
4 '사용여부'
5 '본사실재고'
6 '본사전산'
7 '본사\n실-전'
